In [1]:
print("Initialisation du notebook V5")

Initialisation du notebook V5


# Train Kaggle - Model V5

# PlankEye — DATA_GAN — 2×T4 — ImageNet au départ + reprise automatique

Ce notebook entraîne **PlankEye, le modèle de détection des planches et de leurs 4 coins**.

Le nouveau dataset est `data_gan` (réel + images créées par le GAN), mais le modèle entraîné ici **n'est pas un GAN**.

## Fonctionnement

### Premier lancement
S'il n'existe aucun checkpoint compatible :

- architecture PlankEye v5 ;
- backbone **MobileNetV3-Large** ;
- poids **ImageNet préentraînés** ;
- aucun ancien checkpoint PlankEye ;
- aucun warm-start ;
- départ à l'epoch 1.

### Relance
Si `last_checkpoint_v5.pt` existe et appartient à l'expérience `v5_run` :

- les poids complets PlankEye sont restaurés ;
- l'optimiseur est restauré ;
- l'EMA et le GradScaler sont restaurés ;
- les historiques sont restaurés ;
- l'entraînement reprend à `dernier_epoch + 1`.

### Sauvegarde
- `last` : à chaque epoch en local ;
- `best` : à chaque amélioration ;
- `best + last` : sauvegarde persistante Kaggle tous les 5 epochs ;
- nouvelle session : recherche automatique du checkpoint dans `/kaggle/input`, puis tentative de téléchargement depuis `max778/chekpoints-backbone18`.


## 1 — Vérifier les 2 GPU, Internet et la CLI Kaggle
## 2 — Cloner le dépôt GitHub privé

Cette cellule récupère automatiquement la dernière version de :

```text
maaxxe/Train_Kaggle_plank_Detector
└── model/
    └── model_v5.py
```

Le token reste dans **Kaggle Secrets** et n'est jamais affiché.


In [2]:
!nvidia-smi

import shutil
import socket
import subprocess
import torch

print("\n=== GPU ===")
print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA absent. Active GPU T4 x2 dans Kaggle.")

n_gpus = torch.cuda.device_count()
print("Nombre de GPU :", n_gpus)

for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)} | "
        f"VRAM={props.total_memory / 1024**3:.2f} Go"
    )

if n_gpus < 2:
    raise RuntimeError(
        "Ce notebook exige 2 GPU. Sélectionne GPU T4 x2 avant de continuer."
    )

print("\nOK : 2 GPU détectés.")

print("\n=== INTERNET ===")
try:
    ip = socket.gethostbyname("api.kaggle.com")
    print("api.kaggle.com ->", ip)
except Exception as exc:
    raise RuntimeError(
        "Internet/DNS Kaggle indisponible. Active Internet dans Settings."
    ) from exc

print("\n=== CLI KAGGLE ===")
kaggle_exe = shutil.which("kaggle")
if kaggle_exe is None:
    raise RuntimeError("Commande 'kaggle' introuvable.")

print("CLI :", kaggle_exe)
result = subprocess.run(
    [kaggle_exe, "datasets", "list", "-m"],
    text=True,
    capture_output=True,
)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("La CLI Kaggle n'est pas authentifiée correctement.")

print("Authentification Kaggle : OK")
print("\n".join(result.stdout.splitlines()[:6]))

Wed Sep  9 05:59:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2 — Cloner le dépôt GitHub privé

Cette cellule récupère automatiquement la dernière version de :

```text
maaxxe/Train_Kaggle_plank_Detector
└── model/
    └── model_v5.py
```

Le token reste dans **Kaggle Secrets** et n'est jamais affiché.


In [3]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import subprocess
import shutil
import os

# ============================================================
# CONFIG
# ============================================================

GITHUB_USERNAME = "maaxxe"
GITHUB_REPO = "Train_Kaggle_plank_Detector"

GITHUB_REPO_DIR = Path(
    "/kaggle/working/Train_Kaggle_plank_Detector"
)

print("=" * 72)
print("CLONE GITHUB PRIVÉ")
print("=" * 72)

# ============================================================
# TOKEN GITHUB
# ============================================================

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

if not token:
    raise RuntimeError(
        "Secret GITHUB_TOKEN introuvable. "
        "Ajoute-le dans Kaggle > Add-ons/Secrets et active-le pour ce notebook."
    )

# ============================================================
# GIT ASKPASS
# ============================================================

askpass = Path("/tmp/github_askpass.sh")

askpass.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) echo "$GITHUB_USERNAME" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8",
)

askpass.chmod(0o700)

env = os.environ.copy()
env["GITHUB_USERNAME"] = GITHUB_USERNAME
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

repo_url = (
    f"https://github.com/"
    f"{GITHUB_USERNAME}/{GITHUB_REPO}.git"
)

# ============================================================
# SUPPRESSION ANCIEN CLONE
# ============================================================

if GITHUB_REPO_DIR.exists():
    print("Suppression de l'ancien clone...")
    shutil.rmtree(GITHUB_REPO_DIR)

print("Clone       :", repo_url)
print("Destination :", GITHUB_REPO_DIR)

# ============================================================
# CLONE
# ============================================================

try:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            repo_url,
            str(GITHUB_REPO_DIR),
        ],
        env=env,
        check=True,
    )

finally:
    if askpass.exists():
        askpass.unlink()

    env.pop("GITHUB_TOKEN", None)
    del token

# ============================================================
# AFFICHAGE ARBORESCENCE
# ============================================================

def print_tree(path: Path, prefix="", max_depth=5, depth=0):

    if depth >= max_depth:
        return

    try:
        entries = sorted(
            path.iterdir(),
            key=lambda p: (p.is_file(), p.name.lower())
        )
    except PermissionError:
        return

    for i, entry in enumerate(entries):

        if entry.name == ".git":
            continue

        is_last = i == len(entries) - 1

        connector = "└── " if is_last else "├── "

        print(prefix + connector + entry.name)

        if entry.is_dir():

            extension = "    " if is_last else "│   "

            print_tree(
                entry,
                prefix + extension,
                max_depth=max_depth,
                depth=depth + 1,
            )


print()
print("=" * 72)
print("ARBORESCENCE DU REPO")
print("=" * 72)

print(GITHUB_REPO_DIR.name)
print_tree(GITHUB_REPO_DIR)

# ============================================================
# RECHERCHE AUTOMATIQUE DES model_v5.py
# ============================================================

print()
print("=" * 72)
print("FICHIERS model_v5.py TROUVÉS")
print("=" * 72)

model_files = list(
    GITHUB_REPO_DIR.rglob("model_v5.py")
)

if not model_files:
    print("Aucun model_v5.py trouvé.")
else:
    for model in model_files:
        print(model)

print()
print("Repo OK :", GITHUB_REPO_DIR.exists())
print("\nGitHub privé prêt.")

CLONE GITHUB PRIVÉ
Clone       : https://github.com/maaxxe/Train_Kaggle_plank_Detector.git
Destination : /kaggle/working/Train_Kaggle_plank_Detector


Cloning into '/kaggle/working/Train_Kaggle_plank_Detector'...



ARBORESCENCE DU REPO
Train_Kaggle_plank_Detector
├── GAN_PlankEYE
│   ├── config.py
│   ├── dataset.py
│   ├── GAN_PlankEye_v2_Kaggle_512.ipynb
│   ├── generate.py
│   ├── models.py
│   ├── prepare_dataset.py
│   ├── README.md
│   ├── train.py
│   └── utils.py
├── GeoNet
│   ├── model
│   │   └── multiforme_model.py
│   └── train
│       └── Train_GeoNet_Kaggle_2xT4.ipynb
├── GeoNet_paddle
│   ├── model
│   │   ├── __init__.py
│   │   └── multiforme_model.py
│   ├── model_poids
│   │   └── .gitkeep
│   ├── tools
│   │   └── export_torch_checkpoint_npz.py
│   ├── train
│   │   ├── train_geonet_baidu.py
│   │   └── Train_GeoNet_Baidu_V100.ipynb
│   └── README.md
├── Plankeye
│   ├── dataset
│   │   ├── model_improved.py
│   │   └── PlankEye_Colab.ipynb
│   ├── model
│   │   ├── model.py
│   │   ├── model_v3_.py
│   │   └── model_v5.py
│   ├── model_poids
│   │   ├── best_plankeye_v4_1class_512.pt
│   │   └── last_plankeye_v4_1class_512.pt
│   ├── train
│   │   ├── PlankEye_Training_2xT4

## 3 — Préparer le projet et le nouveau dataset `data_gan`

Cette cellule :

- copie `Plankeye/model/model_v5.py` depuis ton dépôt GitHub privé ;
- cherche automatiquement le nouveau dataset `data_gan` dans `/kaggle/input` ;
- accepte un dataset dont le dossier racine contient directement `images/` et `labels/` ;
- vérifie les associations image/label ;
- **n'importe aucun ancien checkpoint** ;
- prépare un nouvel entraînement indépendant.


In [4]:
from pathlib import Path
import os
import sys
import shutil
import re

INPUT_ROOT = Path("/kaggle/input")
PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")

GITHUB_REPO_DIR = Path(
    "/kaggle/working/Train_Kaggle_plank_Detector"
)
GITHUB_MODEL = (
    GITHUB_REPO_DIR
    / "Plankeye"
    / "model"
    / "model_v5.py"
)
LOCAL_MODEL = PROJECT / "model_v5.py"

# Le reste du notebook utilisera ce lien.
DATA_DIR = PROJECT / "data_kaggle_2"

PROJECT.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PRÉPARATION PLANK EYE — data_kaggle_2 — FROM SCRATCH")
print("=" * 80)


# ============================================================
# 1. MODÈLE DEPUIS GITHUB
# ============================================================

print("\n=== MODÈLE ===")

if not GITHUB_REPO_DIR.exists():
    raise FileNotFoundError(
        "Repo GitHub privé introuvable :\n"
        f"{GITHUB_REPO_DIR}\n\n"
        "Exécute d'abord la cellule de clone GitHub."
    )

if not GITHUB_MODEL.exists():
    raise FileNotFoundError(
        "model/model_v5.py introuvable :\n"
        f"{GITHUB_MODEL}"
    )

shutil.copy2(GITHUB_MODEL, LOCAL_MODEL)

print("Source :", GITHUB_MODEL)
print("Copié  :", LOCAL_MODEL)
print("OK     :", LOCAL_MODEL.exists())


# ============================================================
# 2. TROUVER data_kaggle_2
# ============================================================

print("\n=== DATASET data_kaggle_2 ===")

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

dataset_candidates = []

# Cherche tous les dossiers contenant images/ + labels/.
for images_dir in INPUT_ROOT.rglob("images"):
    if not images_dir.is_dir():
        continue

    candidate = images_dir.parent
    labels_dir = candidate / "labels"

    if not labels_dir.is_dir():
        continue

    path_lower = "/".join(candidate.parts).lower()

    # Priorité aux chemins dont le nom rappelle data_kaggle_2
    score = 0
    if candidate.name.lower() == "data_kaggle_2":
        score += 100
    if "data_kaggle_2" in path_lower:
        score += 80
    if "data_kaggle" in path_lower:
        score += 50
    if "gan" in path_lower:
        score += 20

    dataset_candidates.append((score, candidate))

if not dataset_candidates:
    raise FileNotFoundError(
        "Aucun dataset avec images/ + labels/ trouvé dans /kaggle/input."
    )

dataset_candidates.sort(
    key=lambda x: (-x[0], str(x[1]))
)

print("Candidats compatibles :")
for score, p in dataset_candidates:
    print(f"  score={score:3d} | {p}")

best_score, src = dataset_candidates[0]

if best_score <= 0 and len(dataset_candidates) > 1:
    raise RuntimeError(
        "Plusieurs datasets compatibles trouvés mais aucun ne ressemble "
        "clairement à data_kaggle_2. Vérifie les Inputs Kaggle."
    )

print("\nDataset sélectionné :")
print(" ", src)


# ============================================================
# 3. LIEN /kaggle/working/.../data_kaggle_2
# ============================================================

if DATA_DIR.exists() or DATA_DIR.is_symlink():
    if DATA_DIR.is_symlink():
        DATA_DIR.unlink()
    else:
        shutil.rmtree(DATA_DIR)

DATA_DIR.symlink_to(
    src,
    target_is_directory=True,
)

print("\nLien créé :")
print(" ", DATA_DIR)
print(" ->", src)


# ============================================================
# 4. ASSOCIATION IMAGE <-> LABEL
# Compatible :
# image12.jpg <-> image12.txt
# image12.jpg <-> label12.txt
# ============================================================

def find_label_for_image(img_path: Path):
    p = DATA_DIR / "labels" / f"{img_path.stem}.txt"
    if p.exists():
        return p

    m = re.fullmatch(r"image(\d+)", img_path.stem, flags=re.IGNORECASE)
    if m:
        p = DATA_DIR / "labels" / f"label{m.group(1)}.txt"
        if p.exists():
            return p

    return None


images = sorted(
    p for p in (DATA_DIR / "images").iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
)

labels = sorted((DATA_DIR / "labels").glob("*.txt"))

pairs = []
missing = []

for img in images:
    lbl = find_label_for_image(img)
    if lbl is None:
        missing.append(img)
    else:
        pairs.append((img, lbl))

print("\n" + "=" * 80)
print("VÉRIFICATION DATASET")
print("=" * 80)

print("Images totales :", len(images))
print("Labels totaux  :", len(labels))
print("Paires valides :", len(pairs))
print("Sans label     :", len(missing))

if missing:
    print("\nPremières images sans label :")
    for p in missing[:20]:
        print(" -", p.name)

if not pairs:
    raise RuntimeError(
        "Aucune paire image/label valide trouvée dans data_kaggle_2."
    )

if len(pairs) != len(images):
    raise RuntimeError(
        f"{len(missing)} image(s) n'ont pas de label associé. "
        "Corrige le dataset avant l'entraînement."
    )

print("\n✅ data_kaggle_2 prêt")
print("Images :", DATA_DIR / "images")
print("Labels :", DATA_DIR / "labels")


# ============================================================
# 5. CHECKPOINT data_kaggle_2 : PREMIER LANCEMENT OU REPRISE
# ============================================================

import subprocess
import torch

RUN_ID = "v5_run"
KAGGLE_CKPT_DATASET = "max778/chekpoints-backbone18"

BEST = PROJECT / "best_checkpoint_v5.pt"
LAST = PROJECT / "last_checkpoint_v5.pt"
SPLIT = PROJECT / "split_v5.json"


def checkpoint_info(path: Path):

    if not path.exists():
        return None

    try:
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

        config = ckpt.get("config") or {}

        return {
            "epoch": int(ckpt.get("epoch", -1)),
            "run_id": config.get("run_id"),
            "pretrained_backbone": config.get("pretrained_backbone"),
        }

    except Exception as exc:
        print("Checkpoint illisible :", path, "|", exc)
        return None


def checkpoint_compatible(path: Path):

    info = checkpoint_info(path)

    return (
        info is not None
        and info["run_id"] == RUN_ID
        and info["pretrained_backbone"] is True
    )


def import_pair(last_src: Path):

    if not checkpoint_compatible(last_src):
        return False

    info = checkpoint_info(last_src)

    shutil.copy2(
        last_src,
        LAST,
    )

    best_src = last_src.parent / BEST.name

    if best_src.exists() and checkpoint_compatible(best_src):
        shutil.copy2(
            best_src,
            BEST,
        )

    print(
        f"✅ Checkpoint importé : epoch {info['epoch']}"
    )

    print(
        f"   reprise prévue : epoch {info['epoch'] + 1}"
    )

    return True


print()
print("=" * 80)
print("RECHERCHE CHECKPOINT data_kaggle_2")
print("=" * 80)

resume_found = False


# ------------------------------------------------------------
# A. LAST déjà présent dans /kaggle/working
# ------------------------------------------------------------

if LAST.exists():

    if checkpoint_compatible(LAST):

        info = checkpoint_info(LAST)

        print(
            f"✅ LAST local compatible : epoch {info['epoch']}"
        )

        resume_found = True

    else:

        ignored = PROJECT / "checkpoints_ignores"
        ignored.mkdir(
            parents=True,
            exist_ok=True,
        )

        dst = ignored / LAST.name

        if dst.exists():
            dst.unlink()

        shutil.move(
            str(LAST),
            str(dst),
        )

        print(
            "⚠️ LAST local incompatible déplacé :",
            dst,
        )


# ------------------------------------------------------------
# B. Chercher dans les Inputs Kaggle
# ------------------------------------------------------------

if not resume_found:

    candidates = []

    for p in INPUT_ROOT.rglob(LAST.name):

        if checkpoint_compatible(p):

            info = checkpoint_info(p)

            candidates.append(
                (info["epoch"], p)
            )


    if candidates:

        candidates.sort(
            key=lambda x: x[0],
            reverse=True,
        )

        resume_found = import_pair(
            candidates[0][1]
        )


# ------------------------------------------------------------
# C. Sinon essayer de télécharger le dataset de checkpoints
# ------------------------------------------------------------

if not resume_found:

    kaggle_exe = shutil.which("kaggle")

    if kaggle_exe:

        dl_dir = Path(
            "/kaggle/working/checkpoint_data_kaggle_2_download"
        )

        if dl_dir.exists():
            shutil.rmtree(dl_dir)

        dl_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        print(
            "Tentative téléchargement :",
            KAGGLE_CKPT_DATASET,
        )

        result = subprocess.run(
            [
                kaggle_exe,
                "datasets",
                "download",
                "-d",
                KAGGLE_CKPT_DATASET,
                "-p",
                str(dl_dir),
                "--unzip",
            ],
            text=True,
            capture_output=True,
            check=False,
        )

        if result.returncode == 0:

            downloaded = []

            for p in dl_dir.rglob(LAST.name):

                if checkpoint_compatible(p):

                    info = checkpoint_info(p)

                    downloaded.append(
                        (info["epoch"], p)
                    )


            if downloaded:

                downloaded.sort(
                    key=lambda x: x[0],
                    reverse=True,
                )

                resume_found = import_pair(
                    downloaded[0][1]
                )

        else:

            print(
                "Pas de checkpoint persistant récupérable "
                "(normal au tout premier lancement)."
            )


print()
print("=" * 80)
print("MODE DE DÉMARRAGE")
print("=" * 80)

print("Run ID :", RUN_ID)
print("Best   :", BEST)
print("Last   :", LAST)
print("Split  :", SPLIT)

if resume_found:

    info = checkpoint_info(LAST)

    print()
    print("✅ REPRISE")
    print("Dernier epoch :", info["epoch"])
    print("Prochain epoch:", info["epoch"] + 1)

else:

    print()
    print("✅ PREMIER LANCEMENT")
    print("Départ        : epoch 1")
    print("Backbone      : resnet18")
    print("Poids initiaux: ImageNet préentraînés")
    print("Ancien PlankEye / warm-start : NON")

PRÉPARATION PLANK EYE — data_kaggle_2 — FROM SCRATCH

=== MODÈLE ===
Source : /kaggle/working/Train_Kaggle_plank_Detector/Plankeye/model/model_v5.py
Copié  : /kaggle/working/PlankEyev2_multipieces/model_v5.py
OK     : True

=== DATASET data_kaggle_2 ===
Candidats compatibles :
  score=230 | /kaggle/input/datasets/max778/plankeye/data_kaggle_2/data_kaggle_2
  score= 50 | /kaggle/input/datasets/max778/plankeye/data_kaggle/data_kaggle
  score= 20 | /kaggle/input/datasets/max778/plankeye/data_gan/data_gan
  score=  0 | /kaggle/input/datasets/max778/plankeye/data_6p/data322_melange
  score=  0 | /kaggle/input/datasets/max778/plankeye/dataset_fusionne/dataset_fusionne

Dataset sélectionné :
  /kaggle/input/datasets/max778/plankeye/data_kaggle_2/data_kaggle_2

Lien créé :
  /kaggle/working/PlankEyev2_multipieces/data_kaggle_2
 -> /kaggle/input/datasets/max778/plankeye/data_kaggle_2/data_kaggle_2

VÉRIFICATION DATASET
Images totales : 7429
Labels totaux  : 7429
Paires valides : 7429
Sans label

## 4 — Vérifier la compatibilité de `model_v5.py`


In [6]:


import sys
import torch

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from model_v5 import NUM_CLASSES, IMG_SIZE, PlankEyeV5

print("NUM_CLASSES :", NUM_CLASSES)
print("IMG_SIZE :", IMG_SIZE)

if NUM_CLASSES != 1:
    raise RuntimeError(
        f"Le notebook attend un modèle 1 classe, NUM_CLASSES={NUM_CLASSES}"
    )

# Test de structure du modèle V5
probe = PlankEyeV5()

required_methods = (
    "forward",
)

missing = [name for name in required_methods if not hasattr(probe, name)]

if missing:
    raise RuntimeError(
        "model_v5.py incompatible. Méthodes absentes : " + ", ".join(missing)
    )

print("model_v5.py (ResNet18) compatible avec succès.")

del probe
if torch.cuda.is_available():
    torch.cuda.empty_cache()

NUM_CLASSES : 1
IMG_SIZE : 512
model_v5.py (ResNet18) compatible avec succès.


## 5 — Écrire le script d'entraînement DDP corrigé

Le script ci-dessous contient la sauvegarde locale + l'upload automatique de `best.pt` et `last.pt` vers `max778/chekpoints-backbone18` tous les 5 epochs.

In [8]:
%%writefile /kaggle/working/PlankEyev2_multipieces/train_plankeye_ddp.py


from __future__ import annotations

import copy
import json
from contextlib import nullcontext
import math
import os
import random
import re
import shutil
import subprocess
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from tqdm.auto import tqdm

PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")
SCRIPT_DIR = PROJECT
DATA_DIR = SCRIPT_DIR / "data_kaggle_2"
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

os.chdir(PROJECT)
if str(PROJECT) not in __import__("sys").path:
    __import__("sys").path.insert(0, str(PROJECT))

from model_v5 import (
    IMG_SIZE,
    NUM_CLASSES,
    PlankEyeV5,
    combined_loss_v5,
)

try:
    from model_v5 import compute_detection_metrics
except ImportError:
    def compute_detection_metrics(dets, gts, num_classes):
        return {"map50": 0.0, "map75": 0.0, "pck4": 0.0, "match_recall": 0.0}

CLASS_NAMES = ["plank"]

if NUM_CLASSES != 1:
    raise RuntimeError(f"Le modèle doit être en 1 classe, NUM_CLASSES={NUM_CLASSES}")

SEED = 48
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

BATCH_SIZE = 16
VAL_BATCH_MULT = 2
WORLD_SIZE_ENV = max(1, int(os.environ.get("WORLD_SIZE", "1")))
GRAD_ACCUM = 1 if WORLD_SIZE_ENV > 1 else 2
EPOCHS = 180

LR_HEAD = 3.0e-4
LR_BACKBONE = 2.0e-5
LR_MIN_HEAD = 2.0e-6
LR_MIN_BACKBONE = 2.0e-7
HEAD_WARMUP_EPOCHS = 3
BACKBONE_WARMUP_EPOCHS = 3

UNFREEZE_LAST_EPOCH = 8
UNFREEZE_LAST_N_BLOCKS = 6
UNFREEZE_ALL_EPOCH = 20

WEIGHT_DECAY = 2.0e-4
GRAD_CLIP_NORM = 2.0
EMA_DECAY = 0.9998
EARLY_STOP_PATIENCE = 32
EARLY_STOP_MIN_DELTA = 1e-4

CLASS_BALANCE_POWER = 0.35
CLASS_WEIGHT_MIN = 0.70
CLASS_WEIGHT_MAX = 1.50

ROT_MAX_DEG = 30.0
ROT_MAX_OUT_OF_BOUNDS = 0.015
RANDOM_ERASE_P = 0.15

BEST_MODEL_PATH = SCRIPT_DIR / "best_checkpoint_v5.pt"
LAST_CHECKPOINT_PATH = SCRIPT_DIR / "last_checkpoint_v5.pt"

PERSIST_EVERY = 5
KAGGLE_DATASET_ID = "max778/chekpoints-backbone18"
PERSIST_DIR = Path("/kaggle/working/chekpoints-backbone18_persistent")

PLOT_PATH = SCRIPT_DIR / "training_curves_v5.png"
VIZ_DIR = SCRIPT_DIR / "viz_epochs_v5"
SPLIT_MANIFEST_PATH = SCRIPT_DIR / "split_v5.json"
VIZ_EVERY = 2

RUN_ID = "v5_run"
PRETRAINED_BACKBONE = True

RANK = int(os.environ.get("RANK", "0"))
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))
WORLD_SIZE = max(1, int(os.environ.get("WORLD_SIZE", "1")))
IS_DISTRIBUTED = WORLD_SIZE > 1
IS_MAIN = RANK == 0

if torch.cuda.is_available():
    DEVICE = torch.device(f"cuda:{LOCAL_RANK}")
    DEVICE_TYPE = "cuda"
else:
    DEVICE = torch.device("cpu")
    DEVICE_TYPE = "cpu"

AMP_ENABLED = DEVICE_TYPE == "cuda"


def rank0_print(*args, **kwargs) -> None:
    if IS_MAIN:
        print(*args, **kwargs)


def setup_distributed() -> None:
    if DEVICE_TYPE != "cuda":
        raise RuntimeError("Aucun GPU CUDA détecté. Active un runtime GPU Kaggle.")
    torch.cuda.set_device(LOCAL_RANK)
    if IS_DISTRIBUTED and not dist.is_initialized():
        store_path = "/kaggle/working/PlankEyev2_multipieces/ddp_filestore"
        dist.init_process_group(
            backend="nccl",
            init_method=f"file://{store_path}",
            rank=RANK,
            world_size=WORLD_SIZE,
            device_id=DEVICE,
        )
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


def cleanup_distributed() -> None:
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()


def barrier() -> None:
    if IS_DISTRIBUTED and dist.is_initialized():
        dist.barrier()


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, DDP) else model


def wrap_for_training(model: nn.Module) -> nn.Module:
    if not IS_DISTRIBUTED:
        return model
    return DDP(
        model,
        device_ids=[LOCAL_RANK],
        output_device=LOCAL_RANK,
        broadcast_buffers=False,
        find_unused_parameters=False,
        gradient_as_bucket_view=True,
    )


def broadcast_stop(stop: bool) -> bool:
    if not IS_DISTRIBUTED:
        return bool(stop)
    flag = torch.tensor([1 if stop else 0], device=DEVICE, dtype=torch.int32)
    dist.broadcast(flag, src=0)
    return bool(flag.item())


def set_seed(seed: int, rank: int = 0) -> None:
    process_seed = int(seed) + int(rank) * 1000
    random.seed(process_seed)
    np.random.seed(process_seed)
    torch.manual_seed(process_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(process_seed)


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % 2 ** 32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def reorder_corners(pts: np.ndarray) -> np.ndarray:
    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)
    center = pts.mean(axis=0)
    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
    return pts[np.argsort(angles)].astype(np.float32)


def polygon_area_np(pts: np.ndarray) -> float:
    pts = np.asarray(pts, dtype=np.float64)
    x = pts[:, 0]
    y = pts[:, 1]
    return 0.5 * abs(float(np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)))


def read_label(label_path: Path):
    objects = []
    with open(label_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 9:
                continue
            try:
                coords = np.asarray([float(v) for v in parts[1:]], dtype=np.float32)
            except ValueError:
                continue
            if not np.isfinite(coords).all():
                continue
            pts = coords.reshape(4, 2)
            if (pts < -0.02).any() or (pts > 1.02).any():
                continue
            pts = np.clip(pts, 0.0, 1.0)
            pts = reorder_corners(pts)
            if polygon_area_np(pts) < 1e-06:
                continue
            objects.append({'cls': 0, 'corners': pts})
    return objects


# Fichiers corrompus connus à ignorer
CORRUPTED_IMAGES = {"image4515.jpg"}

def collect_pairs():
    if not IMAGES_DIR.exists() or not LABELS_DIR.exists():
        raise RuntimeError(f'Dataset absent : {IMAGES_DIR} / {LABELS_DIR}')
    pairs = []
    missing = []
    for img_path in sorted(IMAGES_DIR.iterdir()):
        if img_path.suffix.lower() not in IMAGE_EXTS:
            continue
        if img_path.name in CORRUPTED_IMAGES:   # ✅ exclusion immédiate
            continue
        label_path = LABELS_DIR / f'{img_path.stem}.txt'
        if not label_path.exists():
            match = re.fullmatch(r'image(\d+)', img_path.stem, flags=re.IGNORECASE)
            if match:
                alt = LABELS_DIR / f'label{match.group(1)}.txt'
                if alt.exists():
                    label_path = alt
        if label_path.exists():
            pairs.append((img_path, label_path))
        else:
            missing.append(img_path.name)
    if missing:
        raise RuntimeError(f'{len(missing)} image(s) sans label associé.')
    if IS_MAIN:
        print(f"✅ {len(pairs)} paires chargées ({len(CORRUPTED_IMAGES)} image(s) exclue(s) : {CORRUPTED_IMAGES})")
    return pairs
    
def _class_signature(label_path: Path) -> Tuple[int, ...]:
    objects = read_label(label_path)
    return (len(objects),)

def _write_split_manifest(train_p, val_p, test_p) -> None:
    # ✅ Seul rank0 écrit le fichier
    if int(os.environ.get("RANK", "0")) != 0:
        return

    SPLIT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        'version': 1,
        'seed': SEED,
        'dataset_dir': str(DATA_DIR),
        'train': [p[0].name for p in train_p],
        'val':   [p[0].name for p in val_p],
        'test':  [p[0].name for p in test_p]
    }
    tmp = SPLIT_MANIFEST_PATH.with_suffix('.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    os.replace(tmp, SPLIT_MANIFEST_PATH)



def _try_load_split_manifest(pairs):
    if not SPLIT_MANIFEST_PATH.exists():
        return None
    try:
        payload = json.loads(SPLIT_MANIFEST_PATH.read_text(encoding='utf-8'))
    except Exception:
        return None
    by_name = {p[0].name: p for p in pairs}
    used = set()
    def resolve(names):
        out = []
        for name in names:
            pair = by_name.get(name)
            if pair is not None:
                out.append(pair)
                used.add(name)
        return out
    train_p = resolve(payload.get('train', []))
    val_p = resolve(payload.get('val', []))
    test_p = resolve(payload.get('test', []))
    new_pairs = [p for p in pairs if p[0].name not in used]
    train_p.extend(new_pairs)
    if not val_p or not test_p:
        return None
    return (train_p, val_p, test_p)


def split_pairs(pairs):
    existing = _try_load_split_manifest(pairs)
    if existing is not None:
        return existing

    # ── calcul du split (inchangé) ──────────────────────────────────────────
    groups = defaultdict(list)
    for pair in pairs:
        groups[_class_signature(pair[1])].append(pair)

    rng = random.Random(SEED)
    train_p, val_p, test_p = [], [], []

    for signature in sorted(groups, key=str):
        group = sorted(groups[signature])
        rng.shuffle(group)
        n = len(group)
        n_val  = int(round(n * VAL_RATIO))
        n_test = int(round(n * TEST_RATIO))
        if n >= 10:
            n_val  = max(1, n_val)
            n_test = max(1, n_test)
        while n_val + n_test >= n and (n_val > 0 or n_test > 0):
            if n_test >= n_val and n_test > 0:
                n_test -= 1
            elif n_val > 0:
                n_val -= 1
        val_p.extend(group[:n_val])
        test_p.extend(group[n_val:n_val + n_test])
        train_p.extend(group[n_val + n_test:])

    rng.shuffle(train_p)
    rng.shuffle(val_p)
    rng.shuffle(test_p)

    # ✅ rank0 écrit, puis tout le monde attend avant de continuer
    _write_split_manifest(train_p, val_p, test_p)
    if dist.is_initialized():
        dist.barrier()

    return (train_p, val_p, test_p)

def collate_fn(batch):
    return (torch.stack([x[0] for x in batch]), [x[1] for x in batch])


def letterbox_resize(image: Image.Image, target_size: int, objects, return_valid_box=False):
    w, h = image.size
    scale = min(target_size / w, target_size / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    scale_x = new_w / w
    scale_y = new_h / h
    left = (target_size - new_w) // 2
    top = (target_size - new_h) // 2
    resampling = getattr(Image, 'Resampling', Image)
    resized = image.resize((new_w, new_h), resample=resampling.BILINEAR)
    canvas = Image.new('RGB', (target_size, target_size), (114, 114, 114))
    canvas.paste(resized, (left, top))
    transformed = []
    for obj in objects:
        pts = np.asarray(obj['corners'], dtype=np.float32).copy()
        px = pts[:, 0] * w * scale_x + left
        py = pts[:, 1] * h * scale_y + top
        out = np.stack([px / target_size, py / target_size], axis=1)
        transformed.append({'cls': int(obj['cls']), 'corners': reorder_corners(out)})
    if return_valid_box:
        return (canvas, transformed, (left, top, left + new_w, top + new_h))
    return (canvas, transformed)


class GeometricAug:
    def __init__(self, hflip: float=0.5, vflip: float=0.0, rot_deg: float=ROT_MAX_DEG, max_out_of_bounds: float=ROT_MAX_OUT_OF_BOUNDS):
        self.hflip = hflip
        self.vflip = vflip
        self.rot_deg = rot_deg
        self.max_out_of_bounds = max_out_of_bounds

    @staticmethod
    def _rotate_points_pixel(pts_norm: np.ndarray, w: int, h: int, angle_deg: float):
        pts = pts_norm.astype(np.float64).copy()
        x = pts[:, 0] * w
        y = pts[:, 1] * h
        cx, cy = (w / 2.0, h / 2.0)
        dx, dy = (x - cx, y - cy)
        a = math.radians(angle_deg)
        cos_a, sin_a = (math.cos(a), math.sin(a))
        xr = cos_a * dx + sin_a * dy + cx
        yr = -sin_a * dx + cos_a * dy + cy
        return np.stack([xr / w, yr / h], axis=1).astype(np.float32)

    def __call__(self, image: Image.Image, objects):
        w, h = image.size
        out_img = image
        out_objs = [{'cls': o['cls'], 'corners': np.array(o['corners'], copy=True)} for o in objects]
        if random.random() < self.hflip:
            out_img = out_img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            for obj in out_objs:
                obj['corners'][:, 0] = 1.0 - obj['corners'][:, 0]
        angle = random.uniform(-self.rot_deg, self.rot_deg)
        if abs(angle) < 0.5:
            for obj in out_objs:
                obj['corners'] = reorder_corners(obj['corners'])
            return (out_img, out_objs)
        rotated_objs = []
        reject = False
        for obj in out_objs:
            pts = self._rotate_points_pixel(obj['corners'], w, h, angle)
            overflow = np.clip(-pts, 0, None).sum() + np.clip(pts - 1.0, 0, None).sum()
            if overflow > self.max_out_of_bounds:
                reject = True
                break
            pts = np.clip(pts, 0.0, 1.0)
            rotated_objs.append({'cls': obj['cls'], 'corners': reorder_corners(pts)})
        if reject:
            for obj in out_objs:
                obj['corners'] = reorder_corners(obj['corners'])
            return (out_img, out_objs)
        resampling = getattr(Image, 'Resampling', Image)
        rotated_img = out_img.rotate(angle, resample=resampling.BILINEAR, expand=False, fillcolor=(114, 114, 114))
        return (rotated_img, rotated_objs)


class PlankDataset(Dataset):
    def __init__(self, pairs, augment=False):
        self.pairs = list(pairs)
        self.augment = augment
        self.geo_aug = GeometricAug() if augment else None
        if augment:
            self.photo_tf = transforms.Compose([
                transforms.RandomApply([transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.035)], p=0.8),
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.15, 1.0))], p=0.12)
            ])
        else:
            self.photo_tf = None
        self.to_tensor = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, label_path = self.pairs[idx]
        try:
            with Image.open(img_path) as im:
                image = im.convert('RGB')
        except Exception:
            # Image corrompue non détectée au scan → on prend la suivante
            return self.__getitem__((idx + 1) % len(self.pairs))
        
        objects = read_label(label_path)
        if self.geo_aug is not None:
            image, objects = self.geo_aug(image, objects)
        if self.photo_tf is not None:
            image = self.photo_tf(image)
        image, objects, valid_box = letterbox_resize(image, IMG_SIZE, objects, return_valid_box=True)
        tensor = self.to_tensor(image)
        return (tensor, objects)


def make_loaders(train_p, val_p, test_p):
    cpu = os.cpu_count() or 4
    workers = max(1, min(4, cpu // max(WORLD_SIZE, 1)))
    pin = DEVICE_TYPE == "cuda"
    common = {
        "num_workers": workers,
        "pin_memory": pin,
        "persistent_workers": False,                          # ✅ False (était workers > 0)
        "collate_fn": collate_fn,
        "worker_init_fn": seed_worker if workers > 0 else None,
        "prefetch_factor": 2 if workers > 0 else None,       # ✅ ajouté
    }
    train_ds = PlankDataset(train_p, augment=True)
    val_ds = PlankDataset(val_p, augment=False)
    test_ds = PlankDataset(test_p, augment=False)
    train_sampler = DistributedSampler(train_ds, num_replicas=WORLD_SIZE, rank=RANK, shuffle=True, seed=SEED) if IS_DISTRIBUTED else None
    generator = torch.Generator()
    generator.manual_seed(SEED + RANK)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=train_sampler is None, sampler=train_sampler, generator=generator if train_sampler is None else None, **common)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * VAL_BATCH_MULT, shuffle=False, **common)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * VAL_BATCH_MULT, shuffle=False, **common)
    
    if IS_MAIN:
        print(f"📦 DataLoaders prêts : {len(train_p)} images d'entraînement, {len(val_p)} en validation.")
    return train_loader, val_loader, test_loader, train_sampler

class ModelEMA:
    def __init__(self, model: nn.Module, decay: float=EMA_DECAY):
        base = unwrap_model(model)
        self.ema = copy.deepcopy(base).eval()
        self.decay = float(decay)
        self.updates = 0
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        self.updates += 1
        model_state = unwrap_model(model).state_dict()
        for key, value in self.ema.state_dict().items():
            src = model_state[key].detach()
            if value.dtype.is_floating_point:
                value.mul_(self.decay).add_(src, alpha=1.0 - self.decay)
            else:
                value.copy_(src)


def set_backbone_trainable(model, epoch):
    """
    Gestion progressive du backbone ResNet18.

    Epoch 1-7  : backbone gele
    Epoch 8-19 : layer3 + layer4 entrainables
    Epoch 20+  : backbone complet entrainable

    Le choix suit directement UNFREEZE_LAST_EPOCH et
    UNFREEZE_ALL_EPOCH definis plus haut.
    """
    base = unwrap_model(model)
    backbone = base.backbone

    # Tout geler d'abord.
    for param in backbone.parameters():
        param.requires_grad_(False)

    # Derniers blocs du ResNet18.
    if epoch >= UNFREEZE_LAST_EPOCH:
        for param in backbone.layer3.parameters():
            param.requires_grad_(True)
        for param in backbone.layer4.parameters():
            param.requires_grad_(True)

    # Backbone complet.
    if epoch >= UNFREEZE_ALL_EPOCH:
        for param in backbone.parameters():
            param.requires_grad_(True)

    trainable = sum(
        p.numel() for p in backbone.parameters() if p.requires_grad
    )
    total = sum(p.numel() for p in backbone.parameters())

    if epoch < UNFREEZE_LAST_EPOCH:
        phase = "GELÉ"
    elif epoch < UNFREEZE_ALL_EPOCH:
        phase = "PARTIEL (layer3+layer4)"
    else:
        phase = "COMPLET"

    rank0_print(
        f"🔓 Backbone epoch {epoch}: {phase} | "
        f"{trainable:,}/{total:,} paramètres entraînables"
    )


def build_optimizer(model):
    base = unwrap_model(model)

    backbone_params = []
    head_params = []

    for name, param in base.named_parameters():
        if not param.requires_grad:
            continue

        if name.startswith("backbone."):
            backbone_params.append(param)
        else:
            head_params.append(param)

    param_groups = []

    if head_params:
        param_groups.append({
            "params": head_params,
            "lr": LR_HEAD,
            "group_name": "head",
        })

    if backbone_params:
        param_groups.append({
            "params": backbone_params,
            "lr": LR_BACKBONE,
            "group_name": "backbone",
        })

    return torch.optim.AdamW(
        param_groups,
        weight_decay=WEIGHT_DECAY,
    )


def run_epoch(model, loader, optimizer=None, scaler=None, epoch=None, total_epochs=None, phase="train", ema=None):
    training = optimizer is not None
    active_model = model if training else (ema.ema if ema is not None else unwrap_model(model))

    if training:
        model.train()

        # model.train() remet tous les modules en mode train.
        # Pour le backbone gele, on remet les couches concernees
        # en eval afin que les BatchNorm ne modifient pas leurs
        # statistiques pendant la phase de gel.
        if epoch is not None:
            base = unwrap_model(model)
            backbone = base.backbone

            if epoch < UNFREEZE_LAST_EPOCH:
                backbone.eval()
            elif epoch < UNFREEZE_ALL_EPOCH:
                backbone.stem.eval()
                backbone.layer1.eval()
                backbone.layer2.eval()
                backbone.layer3.train()
                backbone.layer4.train()
            else:
                backbone.train()
    else:
        active_model.eval()

    totals = defaultdict(float)
    total_images = 0

    bar = tqdm(loader, desc=f"{phase.upper()} {epoch}/{total_epochs}" if epoch else phase.upper(), disable=(training and not IS_MAIN))

    for step, (images, batch_objects) in enumerate(bar):
        images = images.to(DEVICE, non_blocking=True)
        bs = images.shape[0]

        autocast_ctx = torch.amp.autocast(device_type=DEVICE_TYPE, dtype=torch.float16 if DEVICE_TYPE == "cuda" else torch.bfloat16, enabled=AMP_ENABLED)

        if training:
            optimizer.zero_grad(set_to_none=True)
            with autocast_ctx:
                outputs = active_model(images)
                loss, breakdown = combined_loss_v5(outputs, batch_objects, DEVICE)

            if scaler is not None and scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

            if ema is not None:
                ema.update(model)
        else:
            with torch.no_grad(), autocast_ctx:
                outputs = active_model(images)
                loss, breakdown = combined_loss_v5(outputs, batch_objects, DEVICE)

        totals["loss"] += float(loss.detach()) * bs
        for k, v in breakdown.items():
            if isinstance(v, (int, float)):
                totals[k] += float(v) * bs
        total_images += bs

    denom = max(total_images, 1)
    return {k: v / denom for k, v in totals.items()}


def save_ckpt(path: Path, model, optimizer, scaler, ema, epoch, histories, best_quality):
    if not IS_MAIN:
        return
    base = unwrap_model(model)
    payload = {
        "epoch": epoch,
        "model": base.state_dict(),
        "optim": optimizer.state_dict(),
        "ema": ema.ema.state_dict() if ema else None,
        "histories": histories,
        "best_quality": best_quality,
        "config": {"run_id": RUN_ID, "pretrained_backbone": PRETRAINED_BACKBONE}
    }
    torch.save(payload, path)


def persist_checkpoints_to_kaggle(epoch: int) -> bool:
    if not IS_MAIN:
        return False
    try:
        if PERSIST_DIR.exists():
            shutil.rmtree(PERSIST_DIR)
        PERSIST_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(BEST_MODEL_PATH, PERSIST_DIR / BEST_MODEL_PATH.name)
        shutil.copy2(LAST_CHECKPOINT_PATH, PERSIST_DIR / LAST_CHECKPOINT_PATH.name)
        
        metadata = {"title": "chekpoints-backbone18", "id": KAGGLE_DATASET_ID, "licenses": [{"name": "other"}], "isPrivate": True}
        with open(PERSIST_DIR / "dataset-metadata.json", "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2)
            
        kaggle_exe = shutil.which("kaggle")
        if kaggle_exe:
            subprocess.run([kaggle_exe, "datasets", "version", "-p", str(PERSIST_DIR), "-m", f"Auto checkpoint epoch {epoch}", "--delete-old-versions"], check=False)
            print(f"[PERSIST] Checkpoint sauvegardé sur le dataset Kaggle (epoch {epoch}).")
    except Exception as exc:
        print(f"[PERSIST] Erreur : {exc}")
    return True


def get_checkpoint_start_epoch(path: Path) -> int:
    """Retourne l'epoch de reprise sans charger l'etat de l'optimizer."""
    if not path.exists():
        return 1

    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    return int(checkpoint.get("epoch", 0)) + 1


def load_ckpt(model, optimizer, scaler, ema, path: Path):
    empty_histories = {"train_loss": [], "val_loss": [], "quality": []}
    if not path.exists():
        return 1, empty_histories, float("-inf")

    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    unwrap_model(model).load_state_dict(checkpoint["model"])

    # L'optimizer a deja ete construit avec le bon etat
    # freeze/unfreeze correspondant a l'epoch de reprise.
    if checkpoint.get("optim"):
        try:
            optimizer.load_state_dict(checkpoint["optim"])
        except (ValueError, RuntimeError) as exc:
            rank0_print(
                f"⚠️ Etat optimizer incompatible avec la nouvelle phase: {exc}"
            )
            rank0_print("   → Reprise des poids OK, optimizer réinitialisé pour cette phase.")

    if ema and checkpoint.get("ema"):
        ema.ema.load_state_dict(checkpoint["ema"])

    return (
        int(checkpoint.get("epoch", 0)) + 1,
        checkpoint.get("histories", empty_histories),
        float(checkpoint.get("best_quality", float("-inf"))),
    )


def main():
    setup_distributed()
    try:
        set_seed(SEED, rank=RANK)
        
        if IS_MAIN:
            print("=" * 60)
            print("🚀 DÉMARRAGE DE L'ENTRAÎNEMENT PLANKEYE V5")
            print("=" * 60)

        pairs = collect_pairs()
        train_p, val_p, test_p = split_pairs(pairs)
        barrier()

        train_loader, val_loader, test_loader, train_sampler = make_loaders(train_p, val_p, test_p)

        if IS_MAIN:
            print("📥 Téléchargement / Initialisation du backbone ResNet18 (Poids ImageNet préentraînés)...")
        
        raw_model = PlankEyeV5().to(DEVICE)
        
        if IS_MAIN:
            print("✅ Modèle ResNet18 et têtes V5 initialisés avec succès sur le device :", DEVICE)

        # Déterminer la phase AVANT de construire l'optimizer.
        # Cela permet une reprise propre même si le dernier checkpoint
        # a été sauvegardé pendant une phase partielle ou complète.
        resume_epoch = get_checkpoint_start_epoch(LAST_CHECKPOINT_PATH)
        set_backbone_trainable(raw_model, resume_epoch)

        optimizer = build_optimizer(raw_model)
        scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
        ema = ModelEMA(raw_model)

        start_epoch, histories, best_quality = load_ckpt(
            raw_model, optimizer, scaler, ema, LAST_CHECKPOINT_PATH
        )

        # Sécurité : l'état final doit correspondre exactement à l'epoch reprise.
        set_backbone_trainable(raw_model, start_epoch)
        
        train_model = wrap_for_training(raw_model)

        if start_epoch > 1:
            print(f"🔄 Reprise de l'entraînement à partir de l'epoch {start_epoch}")

        for epoch in range(start_epoch, EPOCHS + 1):
            if train_sampler:
                train_sampler.set_epoch(epoch)

            # Reconfiguration uniquement aux transitions de phase.
            # L'optimizer est reconstruit afin de ne contenir que les
            # paramètres actuellement entraînables.
            if epoch in (UNFREEZE_LAST_EPOCH, UNFREEZE_ALL_EPOCH):
                set_backbone_trainable(raw_model, epoch)
                optimizer = build_optimizer(raw_model)
                rank0_print(f"🔧 Optimizer reconstruit pour la phase de l'epoch {epoch}")

            train_stats = run_epoch(train_model, train_loader, optimizer=optimizer, scaler=scaler, epoch=epoch, total_epochs=EPOCHS, phase="train", ema=ema)
            barrier()
    
            if IS_MAIN:
                val_stats = run_epoch(raw_model, val_loader, optimizer=None, scaler=scaler, epoch=epoch, total_epochs=EPOCHS, phase="val", ema=ema)
                
                histories["train_loss"].append(train_stats.get("loss", 0.0))
                histories["val_loss"].append(val_stats.get("loss", 0.0))
                q = -val_stats.get("loss", float("inf"))
                histories["quality"].append(q)

                # ✅ Résumé de l'epoch
                print(
                    f"📊 Epoch {epoch}/{EPOCHS} | "
                    f"Train loss: {train_stats.get('loss', 0):.4f} | "
                    f"Val loss: {val_stats.get('loss', 0):.4f} | "
                    f"Hmap: {val_stats.get('heatmap', 0):.4f} | "
                    f"Corner: {val_stats.get('corner', 0):.4f} | "
                    f"Geom: {val_stats.get('geometry', 0):.4f}"
                )
    
                if q > best_quality:
                    best_quality = q
                    save_ckpt(BEST_MODEL_PATH, raw_model, optimizer, scaler, ema, epoch, histories, best_quality)
                    print(f"⭐ Nouveau meilleur modèle sauvegardé à l'epoch {epoch} (Qualité: {q:.4f})")
    
                save_ckpt(LAST_CHECKPOINT_PATH, raw_model, optimizer, scaler, ema, epoch, histories, best_quality)
    
                if epoch % PERSIST_EVERY == 0:
                    persist_checkpoints_to_kaggle(epoch)
    
            barrier()  # ✅ rank1 attend que rank0 finisse validation + sauvegarde
    finally:
        cleanup_distributed()

if __name__ == "__main__":
    main()
    


Overwriting /kaggle/working/PlankEyev2_multipieces/train_plankeye_ddp.py


## 6 — Vérifier la syntaxe et les réglages critiques


In [9]:
from pathlib import Path

TRAIN_SCRIPT = Path(
    "/kaggle/working/PlankEyev2_multipieces/train_plankeye_ddp.py"
)

source = TRAIN_SCRIPT.read_text(encoding="utf-8")
compile(source, str(TRAIN_SCRIPT), "exec")

checks = {
    "BATCH_SIZE = 16": "BATCH_SIZE = 16" in source,
    "find_unused_parameters=False": "find_unused_parameters=False" in source,
    "FileStore": 'init_method=f"file://{store_path}"' in source,
    "PERSIST_EVERY = 5": "PERSIST_EVERY = 5" in source,
    "CLI directe kaggle": 'shutil.which("kaggle")' in source,
}

print("Syntaxe OK :", TRAIN_SCRIPT)
print("Nombre de lignes :", len(source.splitlines()))
print()

for name, ok in checks.items():
    print(f"{name:<34} -> {'OK' if ok else 'MANQUANT'}")

if not all(checks.values()):
    raise RuntimeError("Un réglage critique manque dans le script.")

Syntaxe OK : /kaggle/working/PlankEyev2_multipieces/train_plankeye_ddp.py
Nombre de lignes : 857

BATCH_SIZE = 16                    -> OK
find_unused_parameters=False       -> OK
FileStore                          -> OK
PERSIST_EVERY = 5                  -> OK
CLI directe kaggle                 -> OK


## mettre batch size 12 apres epoch 8 puis 8 apres epoch 20

## 7 — Vérifier le mode de démarrage


In [10]:
from pathlib import Path
import torch

PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")

RUN_ID = "v5_run"

LAST = PROJECT / "last_checkpoint_v5.pt"
BEST = PROJECT / "best_checkpoint_v5.pt"

print("=" * 72)
print("MODE DE DÉMARRAGE")
print("=" * 72)

if LAST.exists():
    ckpt = torch.load(LAST, map_location="cpu", weights_only=False)
    config = ckpt.get("config") or {}
    compatible = (config.get("run_id") == RUN_ID and config.get("pretrained_backbone") is True)
else:
    ckpt = None
    compatible = False

if compatible:
    epoch = int(ckpt.get("epoch", 0))
    print("✅ REPRISE")
    print("Dernier epoch :", epoch)
    print("Prochain epoch:", epoch + 1)
else:
    print("✅ PREMIER LANCEMENT")
    print("Départ        : epoch 1")

print()
print("LAST :", LAST)
print("BEST :", BEST)
## 8 — Lancer l'entraînement sur **2 GPU**


MODE DE DÉMARRAGE
✅ REPRISE
Dernier epoch : 7
Prochain epoch: 8

LAST : /kaggle/working/PlankEyev2_multipieces/last_checkpoint_v5.pt
BEST : /kaggle/working/PlankEyev2_multipieces/best_checkpoint_v5.pt


In [11]:
import os
import torch

LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))

DEVICE = torch.device(
    f"cuda:{LOCAL_RANK}" if torch.cuda.is_available() else "cpu"
)

raw_model = PlankEyeV5().to(DEVICE)


def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable


# ─────────────────────────────────────────────
# TOTAL
# ─────────────────────────────────────────────
total, trainable = count_params(raw_model)

print("=" * 65)
print("📊 PARAMÈTRES PLANKEYE V5")
print("=" * 65)

print(f"Device                  : {DEVICE}")
print(f"Paramètres totaux       : {total:,}")
print(f"Paramètres entraînables : {trainable:,}")
print(f"Paramètres gelés        : {total - trainable:,}")

print()
print("-" * 65)
print("📦 PARAMÈTRES PAR COMPOSANT")
print("-" * 65)


# ─────────────────────────────────────────────
# BACKBONE
# ─────────────────────────────────────────────
if hasattr(raw_model, "backbone"):
    backbone_total, backbone_trainable = count_params(raw_model.backbone)

    print(
        f"Backbone                : "
        f"{backbone_total:,} "
        f"(trainables: {backbone_trainable:,}) "
        f"[{100 * backbone_total / total:.2f}%]"
    )


# ─────────────────────────────────────────────
# RESTE DU MODÈLE = HEAD / NECK
# ─────────────────────────────────────────────
backbone_ids = {
    id(p) for p in raw_model.backbone.parameters()
} if hasattr(raw_model, "backbone") else set()

head_params = [
    p for p in raw_model.parameters()
    if id(p) not in backbone_ids
]

head_total = sum(p.numel() for p in head_params)
head_trainable = sum(
    p.numel() for p in head_params
    if p.requires_grad
)

print(
    f"Head / Neck             : "
    f"{head_total:,} "
    f"(trainables: {head_trainable:,}) "
    f"[{100 * head_total / total:.2f}%]"
)

print("-" * 65)


# ─────────────────────────────────────────────
# DÉTAIL DES SOUS-MODULES DIRECTS
# ─────────────────────────────────────────────
print()
print("🔍 DÉTAIL DES SOUS-MODULES")
print("-" * 65)

for name, module in raw_model.named_children():
    n_total, n_trainable = count_params(module)

    print(
        f"{name:<25} : "
        f"{n_total:>12,} "
        f"| trainable: {n_trainable:>12,} "
        f"| {100 * n_total / total:>6.2f}%"
    )

print("=" * 65)

📊 PARAMÈTRES PLANKEYE V5
Device                  : cuda:0
Paramètres totaux       : 11,970,059
Paramètres entraînables : 11,970,059
Paramètres gelés        : 0

-----------------------------------------------------------------
📦 PARAMÈTRES PAR COMPOSANT
-----------------------------------------------------------------
Backbone                : 11,176,512 (trainables: 11,176,512) [93.37%]
Head / Neck             : 793,547 (trainables: 793,547) [6.63%]
-----------------------------------------------------------------

🔍 DÉTAIL DES SOUS-MODULES
-----------------------------------------------------------------
backbone                  :   11,176,512 | trainable:   11,176,512 |  93.37%
lat_s4                    :        8,320 | trainable:        8,320 |   0.07%
lat_s8                    :       16,512 | trainable:       16,512 |   0.14%
lat_s16                   :       32,896 | trainable:       32,896 |   0.27%
lat_s32                   :       65,664 | trainable:       65,664 |   0.55%
r

## 8 — Lancer l'entraînement sur **2 GPU**


In [12]:
import os
import sys
import subprocess
import torch
from pathlib import Path

PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")
TRAIN_SCRIPT = PROJECT / "train_plankeye_ddp.py"
STORE_FILE = PROJECT / "ddp_filestore"

if not TRAIN_SCRIPT.exists():
    raise FileNotFoundError(TRAIN_SCRIPT)

if torch.cuda.device_count() < 2:
    raise RuntimeError("2 GPU requis. Active GPU T4 x2.")

if STORE_FILE.exists():
    STORE_FILE.unlink()

print("Lancement DDP FileStore : GPU 0 + GPU 1")

base_env = os.environ.copy()
base_env["OMP_NUM_THREADS"] = "2"
base_env["PYTHONUNBUFFERED"] = "1"
base_env["NCCL_SOCKET_IFNAME"] = "lo"
base_env["NCCL_SOCKET_FAMILY"] = "AF_INET"
base_env["NCCL_DEBUG"] = "WARN"

processes = []
for rank in range(2):
    env = base_env.copy()
    env["RANK"] = str(rank)
    env["LOCAL_RANK"] = str(rank)
    env["WORLD_SIZE"] = "2"
    process = subprocess.Popen([sys.executable, str(TRAIN_SCRIPT)], cwd=str(PROJECT), env=env)
    processes.append(process)

return_codes = [p.wait() for p in processes]
if any(code != 0 for code in return_codes):
    raise RuntimeError(f"Un processus DDP a échoué : {return_codes}")

print("Entraînement terminé correctement.")

Lancement DDP FileStore : GPU 0 + GPU 1
============================PlankEye V5 model loaded TEST V2============================
============================PlankEye V5 model loaded TEST V2============================
NCCL version 2.27.5+cuda12.9
🚀 DÉMARRAGE DE L'ENTRAÎNEMENT PLANKEYE V5
✅ 7428 paires chargées (1 image(s) exclue(s) : {'image4515.jpg'})
📦 DataLoaders prêts : 5942 images d'entraînement, 743 en validation.
📥 Téléchargement / Initialisation du backbone ResNet18 (Poids ImageNet préentraînés)...
✅ Modèle ResNet18 et têtes V5 initialisés avec succès sur le device : cuda:0
🔓 Backbone epoch 8: PARTIEL (layer3+layer4) | 10,493,440/11,176,512 paramètres entraînables
⚠️ Etat optimizer incompatible avec la nouvelle phase: loaded state dict has a different number of parameter groups
   → Reprise des poids OK, optimizer réinitialisé pour cette phase.
🔓 Backbone epoch 8: PARTIEL (layer3+layer4) | 10,493,440/11,176,512 paramètres entraînables
🔄 Reprise de l'entraînement à partir de l'e

VAL 8/180: 100%|██████████| 24/24 [00:49<00:00,  2.07s/it]


📊 Epoch 8/180 | Train loss: 0.9703 | Val loss: 5.0373 | Hmap: 4.5101 | Corner: 0.0364 | Geom: 0.1948
⭐ Nouveau meilleur modèle sauvegardé à l'epoch 8 (Qualité: -5.0373)


VAL 9/180: 100%|██████████| 24/24 [00:42<00:00,  1.77s/it]


📊 Epoch 9/180 | Train loss: 0.8574 | Val loss: 5.0076 | Hmap: 4.4976 | Corner: 0.0347 | Geom: 0.1910
⭐ Nouveau meilleur modèle sauvegardé à l'epoch 9 (Qualité: -5.0076)


...

📊 Epoch 22/180 | Train loss: 0.4796 | Val loss: 4.1799 | Hmap: 3.8461 | Corner: 0.0166 | Geom: 0.1621
⭐ Nouveau meilleur modèle sauvegardé à l'epoch 22 (Qualité: -4.1799)


VAL 23/180: 100%|██████████| 24/24 [00:47<00:00,  2.00s/it]


📊 Epoch 23/180 | Train loss: 0.4401 | Val loss: 4.0959 | Hmap: 3.7728 | Corner: 0.0157 | Geom: 0.1590
⭐ Nouveau meilleur modèle sauvegardé à l'epoch 23 (Qualité: -4.0959)


TRAIN 24/180:  10%|█         | 19/186 [04:08<36:28, 13.10s/it] 


KeyboardInterrupt: 

In [1]:
!nvidia-smi


Tue Sep  8 12:51:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!kill -9 <PID>

## 9 — Sauvegarde **manuelle persistante** des checkpoints V5


In [13]:
from pathlib import Path
import shutil
import json
import subprocess
import socket
import torch

PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")
UPLOAD_DIR = Path("/kaggle/working/chekpoints-backbone18_persistent")
KAGGLE_DATASET_ID = "max778/chekpoints-backbone18"

BEST = PROJECT / "best_checkpoint_v5.pt"
LAST = PROJECT / "last_checkpoint_v5.pt"

if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(BEST, UPLOAD_DIR / BEST.name)
shutil.copy2(LAST, UPLOAD_DIR / LAST.name)

ckpt = torch.load(LAST, map_location="cpu", weights_only=False)
epoch = int(ckpt.get("epoch", -1))
metadata = {
    "title": "chekpoints-backbone18",
    "id": KAGGLE_DATASET_ID,
    "licenses": [{"name": "other"}],
    "isPrivate": True,
}
with open(UPLOAD_DIR / "dataset-metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

kaggle_exe = shutil.which("kaggle")
cmd = [kaggle_exe, "datasets", "version", "-p", str(UPLOAD_DIR), "-m", f"Checkpoint V5 epoch {epoch}", "--delete-old-versions"]
result = subprocess.run(cmd, text=True, capture_output=True, check=False)

if result.returncode == 0:
    print("\n✅ Nouvelle version persistante sauvegardée.")
else:
    create_result = subprocess.run([kaggle_exe, "datasets", "create", "-p", str(UPLOAD_DIR)], text=True, capture_output=True, check=False)
    if create_result.returncode != 0:
        raise RuntimeError("Sauvegarde persistante Kaggle impossible.")
    print("\n✅ Dataset de checkpoints créé.")


✅ Nouvelle version persistante sauvegardée.


In [14]:
from pathlib import Path
import shutil
import subprocess
import json
import torch

PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")
PERSIST = Path("/kaggle/working/chekpoints-backbone18_persistent")
DATASET_ID = "max778/chekpoints-backbone18"

# Recrée le dossier avec les derniers checkpoints
if PERSIST.exists():
    shutil.rmtree(PERSIST)
PERSIST.mkdir(parents=True)

# Copie les checkpoints
for name in [
    "best_checkpoint_v5.pt",
    "last_checkpoint_v5.pt",
]:
    src = PROJECT / name
    if src.exists():
        shutil.copy2(src, PERSIST / name)

# Récupère l'epoch
ckpt = torch.load(
    PROJECT / "last_checkpoint_v5.pt",
    map_location="cpu",
    weights_only=False
)
epoch = int(ckpt.get("epoch", -1))

# Metadata
with open(PERSIST / "dataset-metadata.json", "w") as f:
    json.dump({
        "title": "chekpoints-backbone18",
        "id": DATASET_ID,
        "licenses": [{"name": "other"}],
        "isPrivate": True
    }, f, indent=2)

# Nouvelle VERSION Kaggle
# IMPORTANT : aucune suppression des anciennes versions
kaggle = shutil.which("kaggle")

result = subprocess.run([
    kaggle,
    "datasets", "version",
    "-p", str(PERSIST),
    "-m", f"PlankEye V5 - epoch {epoch}"
], capture_output=True, text=True)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("❌ Échec sauvegarde Kaggle")

print(f"✅ Historique persistant sauvegardé — epoch {epoch}")
print(f"📦 {DATASET_ID}")
print("📚 Anciennes versions conservées")

✅ Historique persistant sauvegardé — epoch 23
📦 max778/chekpoints-backbone18
📚 Anciennes versions conservées


In [8]:
from pathlib import Path

print("Datasets disponibles :\n")

for p in Path("/kaggle/input").iterdir():
    print("📁", p)

print("\nRecherche des checkpoints :\n")

for p in Path("/kaggle/input").rglob("*.pt"):
    print("✅", p)

Datasets disponibles :

📁 /kaggle/input/datasets

Recherche des checkpoints :

✅ /kaggle/input/datasets/max778/chekpoints-backbone18/best_checkpoint_v5.pt
✅ /kaggle/input/datasets/max778/chekpoints-backbone18/last_checkpoint_v5.pt


KeyboardInterrupt: 

In [9]:
from pathlib import Path
import torch

ckpt_path = Path(
    "/kaggle/input/datasets/max778/chekpoints-backbone18/best_checkpoint_v5.pt"
)

print("Checkpoint :", ckpt_path)
print("Existe :", ckpt_path.exists())
print("Taille :", ckpt_path.stat().st_size / 1024**2, "MB")

ckpt = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False
)

print("\nType :", type(ckpt))

if isinstance(ckpt, dict):
    print("\nClés du checkpoint :")
    for k in ckpt.keys():
        print(" -", k)

    if "histories" in ckpt:
        print("\n✅ HISTORIES TROUVÉES !")

        histories = ckpt["histories"]

        print("\nContenu :")
        for k, v in histories.items():
            print(f" - {k}: {len(v)} valeurs")

    else:
        print("\n❌ Pas de 'histories' dans best_checkpoint_v5.pt")

Checkpoint : /kaggle/input/datasets/max778/chekpoints-backbone18/best_checkpoint_v5.pt
Existe : True
Taille : 97.58183193206787 MB

Type : <class 'dict'>

Clés du checkpoint :
 - epoch
 - model
 - optim
 - ema
 - histories
 - best_quality
 - config

✅ HISTORIES TROUVÉES !

Contenu :
 - train_loss: 7 valeurs
 - val_loss: 7 valeurs
 - quality: 7 valeurs


In [10]:
ckpt = torch.load(
    "/kaggle/input/datasets/max778/chekpoints-backbone18/best_checkpoint_v5.pt",
    map_location="cpu",
    weights_only=False
)

histories = ckpt["histories"]

print(histories)

{'train_loss': [2.592055096284423, 1.76401762808264, 1.2696649581645323, 1.1614779813327358, 1.0965146618794608, 1.0667845360641326, 1.0388233130205284], 'val_loss': [5.195178056950525, 5.1740646028582855, 5.150284198540216, 5.128622814270882, 5.106835768970474, 5.083573896503063, 5.061931201939949], 'quality': [-5.195178056950525, -5.1740646028582855, -5.150284198540216, -5.128622814270882, -5.106835768970474, -5.083573896503063, -5.061931201939949]}
